# Session 3 — Evaluation and interpretation

Evaluate inferred proteins with pointwise and spatial diagnostics, then interpret molecular landscapes, residual structure, and exploratory clusters.

For workshop reproducibility, select **Runtime → Change runtime type → 2026.04**
(Python 3.12) before running the bootstrap.

Work through the parts in order. Outputs and JSON checkpoints are written to
`MyDrive/ECCB2026/state`, so they survive a Colab runtime reset. If the runtime stops,
rerun the bootstrap cell, inspect the printed completed checkpoints, and jump to the
first unfinished part; each part reloads its required inputs.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

SESSION_REQUIREMENTS = [('anndata', 'anndata==0.11.4'), ('seaborn', 'seaborn==0.13.2'), ('sklearn', 'scikit-learn==1.7.2')]
NEED_DGAT = False

in_colab = importlib.util.find_spec("google.colab") is not None and Path("/content").is_dir()
if in_colab:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    repo_dir = Path("/content/ECCB-2026-Tutorial")
    tutorial_root = repo_dir / "hands-on_tutorial"
    if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)],
            check=True,
        )
    else:
        # A Colab runtime can outlive the notebook tab. Refresh an existing clone
        # so newly opened notebooks do not silently execute stale setup scripts.
        subprocess.run(
            ["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"],
            check=True,
        )

    drive_root = Path("/content/drive/MyDrive/ECCB2026")
    drive_data = drive_root / "assets" / "DGAT_assets" / "data"
    manifest_path = drive_root / "asset_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing {manifest_path}. Run Session 0 before the tutorial.")
    asset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    local_data = tutorial_root / "external" / "DGAT_assets" / "data"
    local_data.mkdir(parents=True, exist_ok=True)
    for filename in ("Tonsil_RNA.h5ad", "Tonsil_ADT.h5ad"):
        source = drive_data / filename
        destination = local_data / filename
        if not source.is_file() or source.stat().st_size == 0:
            raise FileNotFoundError(
                f"Missing {source}. Run Session 0 Drive preparation before the tutorial."
            )
        expected_bytes = asset_manifest["files"][filename]["bytes"]
        if source.stat().st_size != expected_bytes:
            raise IOError(
                f"Drive asset size mismatch for {filename}: expected {expected_bytes}, "
                f"found {source.stat().st_size}. Rerun Session 0."
            )
        if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
            print(f"Copying {filename} from Drive to the Colab VM ...")
            shutil.copy2(source, destination)

    os.environ["DGAT_TUTORIAL_STATE_DIR"] = str(drive_root / "state")

    if NEED_DGAT:
        dgat_dir = tutorial_root / "external" / "DGAT"
        if not (dgat_dir / "utils" / "Preprocessing.py").is_file():
            dgat_dir.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/osmanbeyoglulab/DGAT.git", str(dgat_dir)],
                check=True,
            )

    missing_specs = [
        package_spec
        for import_name, package_spec in SESSION_REQUIREMENTS
        if importlib.util.find_spec(import_name) is None
    ]
    if missing_specs:
        wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"
        online_command = [sys.executable, "-m", "pip", "install", "-q", *missing_specs]
        if wheelhouse.is_dir() and any(wheelhouse.glob("*.whl")):
            print(f"Installing missing packages using the Drive wheel cache: {missing_specs}")
            cached_command = [
                sys.executable, "-m", "pip", "install", "-q", "--no-index",
                "--find-links", str(wheelhouse), *missing_specs,
            ]
            try:
                subprocess.run(cached_command, check=True)
            except subprocess.CalledProcessError:
                print("Wheel cache was incomplete; falling back to PyPI.")
                subprocess.run(online_command, check=True)
        else:
            print(f"Drive wheel cache missing; installing from PyPI: {missing_specs}")
            subprocess.run(online_command, check=True)
        importlib.invalidate_caches()
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "src" / "dgat_tutorial").is_dir():
            tutorial_root = candidate
            break
    else:
        raise FileNotFoundError("Could not locate hands-on_tutorial/ from the current directory.")

os.chdir(tutorial_root)
src_dir = tutorial_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
completed = sorted(path.name for path in paths.checkpoints.glob("session_*/part_*.json"))
print(f"Tutorial root: {paths.root}")
print(f"Persistent state: {paths.checkpoints.parent}")
print("Completed checkpoints:", completed or "none yet")


## Session 3 · Part 1 — Evaluate pointwise prediction accuracy

**Goal:** quantify how well inferred abundance tracks measured abundance for every protein. Published
DGAT reporting emphasized **Spearman correlation and RMSE**; Pearson is retained as a secondary linear
check. Neither metric alone proves spatial concordance—that is Part 2.

This is an evaluation notebook: it requires the matching official Tonsil RNA/ADT assets and never pairs
the committed predictions with any other observations. Before treating Tonsil as held-out accuracy,
confirm with the DGAT authors that Tonsil was excluded from training and checkpoint selection
(`tonsil_held_out` in the prediction metadata).


### 1. Align observed and predicted proteins and read provenance


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_metadata, load_prediction_table
from dgat_tutorial.evaluation import (
    alignment_report,
    corresponding_rna_baseline,
    protein_correlations,
)
from dgat_tutorial.plotting import plot_correlation_bar
from dgat_tutorial.processing import normalize_rna_dgat, prepare_evaluation_proteins

dataset = load_tutorial_data(paths.raw_data)
prediction_path = preferred_prediction_path(paths)
predicted = load_prediction_table(str(prediction_path))
metadata = load_prediction_metadata(prediction_path)
if metadata:
    print(f"Prediction method: {metadata.get('method')}")
    print(f"tonsil_held_out: {metadata.get('tonsil_held_out')}")
    print(f"training_samples: {metadata.get('training_samples')}")
    print(f"Evaluation note: {metadata.get('evaluation_note')}")
    if metadata.get("tonsil_held_out") is not True:
        print(
            "WARNING: tonsil_held_out is not True in the sidecar. "
            "Do not treat these scores as confirmed held-out accuracy."
        )

# DGAT predictions are on the CLR protein scale used in training/evaluation.
# Score against CLR-normalized observed ADT (never raw counts).
observed_raw = dataset.proteins
report = alignment_report(observed_raw, predicted)
print(report)
if report["spots_only_in_observed"] or report["spots_only_in_predicted"]:
    print("WARNING: spot ID sets differ; evaluation uses the intersection and reports mismatches above.")
if report["proteins_only_in_observed"] or report["proteins_only_in_predicted"]:
    print("WARNING: protein panels differ; only shared proteins are scored.")

common_spots = observed_raw.index.intersection(predicted.index)
common_proteins = list(report["proteins_evaluated"])
if not len(common_spots) or not common_proteins:
    raise ValueError("Observed and predicted tables must share spot IDs and canonical protein names.")
observed = prepare_evaluation_proteins(observed_raw.loc[common_spots, common_proteins])
predicted = predicted.loc[common_spots, common_proteins]

scale_summary = pd.DataFrame({
    "observed_clr_mean": observed.mean(),
    "predicted_mean": predicted.mean(),
    "observed_clr_std": observed.std(),
    "predicted_std": predicted.std(),
})
display(scale_summary.head())

correlations = protein_correlations(observed, predicted)
correlations


#### Figure 10 — Accuracy across the complete protein panel (Spearman)


In [ ]:
ax = plot_correlation_bar(correlations, metric="spearman")
ax.set_title("Per-protein Spearman accuracy (DGAT-style)")
correlation_bar_path = paths.figures / "session03_prediction_correlations.png"
plt.tight_layout(); plt.savefig(correlation_bar_path, dpi=160, bbox_inches="tight"); plt.show()

rmse_summary = correlations[["protein", "spearman", "pearson", "rmse"]].copy()
print(
    "Panel summary: "
    f"median Spearman={rmse_summary['spearman'].median():.3f}, "
    f"median Pearson={rmse_summary['pearson'].median():.3f}, "
    f"median RMSE={rmse_summary['rmse'].median():.3f}"
)
display(rmse_summary.head(10))


### 2. Corresponding-RNA baseline (nonspatial)


In [ ]:
# Use the same RNA normalize/scale recipe as DGAT training for a fairer nonspatial baseline.
# Compare ranks (Spearman) primarily; RMSE across RNA vs CLR-protein units remains secondary.
transcripts = normalize_rna_dgat(dataset.transcripts.loc[common_spots])
rna_baseline = corresponding_rna_baseline(transcripts, observed)
rna_correlations = protein_correlations(observed[rna_baseline.columns], rna_baseline)
comparison = (
    correlations.set_index("protein")[["spearman", "rmse"]]
    .join(rna_correlations.set_index("protein")[["spearman", "rmse"]], rsuffix="_rna")
    .rename(columns={"spearman": "spearman_dgat", "rmse": "rmse_dgat",
                     "spearman_rna": "spearman_rna", "rmse_rna": "rmse_rna"})
    .dropna()
    .sort_values("spearman_dgat", ascending=False)
)
print(f"Corresponding-RNA baseline proteins matched: {len(comparison)}")
display(comparison.head(10))
baseline_path = paths.results / "session03_dgat_vs_rna_baseline.csv"
comparison.to_csv(baseline_path)


#### Figure 11 — Observed versus predicted abundance for representative proteins


In [ ]:
ranked = correlations.sort_values("spearman")
representative = list(dict.fromkeys([
    ranked.iloc[-1]["protein"], ranked.iloc[len(ranked)//2]["protein"], ranked.iloc[0]["protein"]
]))
fig, axes = plt.subplots(1, len(representative), figsize=(4 * len(representative), 3.6), squeeze=False)
for ax, protein in zip(axes.ravel(), representative):
    row = correlations.set_index("protein").loc[protein]
    ax.scatter(observed[protein], predicted[protein], s=10, alpha=0.45, color="#3b7a78")
    lims = [
        min(observed[protein].min(), predicted[protein].min()),
        max(observed[protein].max(), predicted[protein].max()),
    ]
    ax.plot(lims, lims, color="black", linewidth=0.8, linestyle="--")
    ax.set(
        xlabel="observed", ylabel="predicted",
        title=f"{protein}\nSpearman={row['spearman']:.2f}; RMSE={row['rmse']:.2f}",
    )
    ax.set_aspect("equal", adjustable="box")
scatter_path = paths.figures / "session03_observed_vs_predicted_scatter.png"
fig.tight_layout(); fig.savefig(scatter_path, dpi=160, bbox_inches="tight"); plt.show()


**How to read it:** a high Spearman with a compressed prediction range can still underestimate
biological extremes (check RMSE and the diagonal). Outliers may be technical, but they can also identify
rare regions worth inspecting spatially. Avoid summarizing a 31-protein panel with only its mean correlation.


In [ ]:
table_path = paths.results / "session03_prediction_correlations.csv"
correlations.to_csv(table_path, index=False)
manifest = write_checkpoint(
    "3.1", [table_path, baseline_path, correlation_bar_path, scatter_path],
    summary={
        "spots": len(common_spots),
        "proteins_evaluated": len(correlations),
        "median_spearman": float(correlations["spearman"].median()),
        "median_rmse": float(correlations["rmse"].median()),
    },
    start=paths.root,
)
print(f"Checkpoint written: {manifest}")


### Check

Name a high-, middle-, and low-performing protein by Spearman/RMSE and describe whether errors look like
noise, range compression, or systematic bias. Note whether DGAT beats the corresponding-RNA baseline.


## Session 3 · Part 2 — Evaluate spatial coherence

**Goal:** test whether DGAT preserves local tissue structure. Comparing univariate Moran's I for
observed versus predicted values shows whether the two matrices have similar *overall autocorrelation*.
That is **not** the same as spatial concordance: a highly smoothed but misplaced prediction can still
match Moran's I. We therefore also report residual Moran's I and a simple bivariate Moran's I between
observed and predicted maps.


### 1. Load aligned observations, predictions, and coordinates


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.evaluation import (
    alignment_report,
    bivariate_morans_i,
    morans_i,
    residual_morans_i,
)

from dgat_tutorial.processing import prepare_evaluation_proteins

dataset = load_tutorial_data(paths.raw_data)
predicted = load_prediction_table(str(preferred_prediction_path(paths)))
report = alignment_report(dataset.proteins, predicted)
print(report)
common_spots = dataset.spots.index.intersection(dataset.proteins.index).intersection(predicted.index)
common_proteins = list(report["proteins_evaluated"])
if not len(common_spots) or not common_proteins:
    raise ValueError("Observed and predicted data do not share both spot IDs and protein names.")
spots = dataset.spots.loc[common_spots]
observed = prepare_evaluation_proteins(dataset.proteins.loc[common_spots, common_proteins])
predicted = predicted.loc[common_spots, common_proteins]


### 2. Calculate observed/predicted Moran's I plus residual and bivariate concordance


In [ ]:
moran_table = pd.DataFrame([
    {
        "protein": protein,
        "observed_morans_i": morans_i(observed[protein], spots),
        "predicted_morans_i": morans_i(predicted[protein], spots),
        "residual_morans_i": residual_morans_i(observed[protein], predicted[protein], spots),
        "bivariate_morans_i": bivariate_morans_i(observed[protein], predicted[protein], spots),
    }
    for protein in common_proteins
])
moran_table["difference"] = moran_table["predicted_morans_i"] - moran_table["observed_morans_i"]
moran_table.sort_values("difference")


#### Figure 12 — Observed versus predicted spatial autocorrelation


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=moran_table, x="observed_morans_i", y="predicted_morans_i", hue="protein", s=70, ax=ax)
limits = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(limits, limits, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Univariate Moran's I (autocorrelation similarity ≠ spatial concordance)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, frameon=False)
moran_scatter_path = paths.figures / "session03_morans_i.png"
fig.tight_layout(); fig.savefig(moran_scatter_path, dpi=160, bbox_inches="tight"); plt.show()


#### Figure 13 — Smoothing bias and residual spatial structure


In [ ]:
ordered = moran_table.sort_values("difference")
fig, axes = plt.subplots(1, 2, figsize=(12, max(4, 0.24 * len(ordered))))
colors = ["#b65f3c" if value < 0 else "#4c78a8" for value in ordered["difference"]]
axes[0].barh(ordered["protein"], ordered["difference"], color=colors)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set(xlabel="predicted Moran's I − observed Moran's I", title="Smoothing bias")
ordered_res = moran_table.sort_values("residual_morans_i")
axes[1].barh(ordered_res["protein"], ordered_res["residual_morans_i"], color="#6b7b8c")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set(xlabel="Moran's I of residuals", title="Spatially structured errors")
difference_path = paths.figures / "session03_morans_i_difference.png"
fig.tight_layout(); fig.savefig(difference_path, dpi=160, bbox_inches="tight"); plt.show()
print(
    "Median bivariate Moran's I (observed vs predicted): "
    f"{moran_table['bivariate_morans_i'].median():.3f}"
)


Positive univariate differences can indicate over-smoothing; negative differences can indicate lost
spatial signal. Residual Moran's I near zero suggests unstructured errors; large residual Moran's I
suggests spatially localized mistakes. Interpretation depends on the neighborhood radius and tissue
geometry—report the rule and compare proteins under the same rule. Stronger tests (permutation p-values,
spatial-lag correlation, neighborhood sensitivity) are recommended before publication.


In [ ]:
table_path = paths.results / "session03_morans_i.csv"
moran_table.to_csv(table_path, index=False)
manifest = write_checkpoint(
    "3.2", [table_path, moran_scatter_path, difference_path],
    summary={
        "proteins_evaluated": len(moran_table),
        "median_bivariate_morans_i": float(moran_table["bivariate_morans_i"].median()),
    },
    start=paths.root,
)
print(f"Checkpoint written: {manifest}")


### Check

Find one protein whose pointwise correlation and spatial-coherence result agree, and one where they tell
different stories. Prefer residual or bivariate Moran when univariate Moran's I alone looks reassuring.


## Session 3 · Part 3 — Interpret inferred molecular landscapes

**Goal:** move from scores to biological and technical interpretation. We select a well-predicted protein
by Spearman, identify the RNA feature most associated with its measured abundance, compare four spatial
maps on shared color scales, and derive an exploratory embedding from the complete inferred protein panel.
Clustering uses a higher-dimensional PC space; PC1/PC2 are for visualization only.


### 1. Select a protein and a transcript using explicit evidence


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.evaluation import protein_correlations
from dgat_tutorial.plotting import plot_spatial_feature
from dgat_tutorial.processing import normalize_rna_dgat, prepare_evaluation_proteins

dataset = load_tutorial_data(paths.raw_data)
predicted = load_prediction_table(str(preferred_prediction_path(paths)))
common_spots = dataset.spots.index.intersection(dataset.proteins.index).intersection(predicted.index)
if common_spots.empty:
    raise ValueError("Observed and predicted data have no shared spot IDs.")
spots = dataset.spots.loc[common_spots]
observed = prepare_evaluation_proteins(dataset.proteins.loc[common_spots])
predicted = predicted.loc[common_spots]
transcripts = normalize_rna_dgat(dataset.transcripts.loc[common_spots])

correlations = protein_correlations(observed, predicted)
protein = correlations.sort_values("spearman", ascending=False).iloc[0]["protein"]
gene_correlations = transcripts.corrwith(observed[protein]).dropna().sort_values(key=abs, ascending=False)
gene = gene_correlations.index[0]
print(f"Selected protein {protein} (best Spearman on CLR scale); associated transcript {gene} (|r| maximum).")


#### Figure 14 — Demo2-inspired RNA, observed-protein, and predicted-protein marker panel

Upstream `Demo2_Predict.ipynb` visualizes PAX5, MS4A1, and PDCD1 RNA alongside DGAT
predictions. Because this Tonsil dataset also includes measured ADT, we add the observed protein
as a third view. Each row follows one marker from RNA to measured protein to inferred protein.


In [ ]:
marker_panel = ["PAX5", "MS4A1", "PDCD1"]
missing_rna = [marker for marker in marker_panel if marker not in transcripts.columns]
missing_observed = [marker for marker in marker_panel if marker not in observed.columns]
missing_predicted = [marker for marker in marker_panel if marker not in predicted.columns]
if missing_rna or missing_observed or missing_predicted:
    raise KeyError(
        "Marker panel is incomplete: "
        f"RNA missing={missing_rna}, observed protein missing={missing_observed}, "
        f"predicted protein missing={missing_predicted}"
    )

fig, axes = plt.subplots(len(marker_panel), 3, figsize=(12, 10), squeeze=False)
for row, marker in enumerate(marker_panel):
    # RNA has a different measurement scale. Observed and predicted protein share a row-wise scale.
    protein_vmin = float(min(observed[marker].min(), predicted[marker].min()))
    protein_vmax = float(max(observed[marker].max(), predicted[marker].max()))
    plot_spatial_feature(
        spots, transcripts[marker], f"{marker} RNA", cmap="magma", ax=axes[row, 0]
    )
    plot_spatial_feature(
        spots, observed[marker], f"Observed {marker} protein", ax=axes[row, 1],
        vmin=protein_vmin, vmax=protein_vmax,
    )
    plot_spatial_feature(
        spots, predicted[marker], f"Predicted {marker} protein", ax=axes[row, 2],
        vmin=protein_vmin, vmax=protein_vmax,
    )
marker_panel_path = paths.figures / "session03_demo2_marker_panel.png"
fig.tight_layout(); fig.savefig(marker_panel_path, dpi=180, bbox_inches="tight"); plt.show()


**How to read it:** compare spatial location rather than raw color between the RNA and protein
columns because their scales differ. Within each row, observed and predicted protein use the same
color limits, so range compression, missing regions, and over-smoothed predictions are visible.
Similar RNA and protein maps are possible, but disagreement can reflect translation, trafficking,
degradation, antibody behavior, or model error.


#### Figure 15 — Transcript, measured protein, inferred protein, and residual


In [ ]:
residual = predicted[protein] - observed[protein]
limit = float(np.abs(residual).max())
shared_vmin = float(min(observed[protein].min(), predicted[protein].min()))
shared_vmax = float(max(observed[protein].max(), predicted[protein].max()))
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
plot_spatial_feature(spots, transcripts[gene], f"Normalized RNA: {gene}", cmap="magma", ax=axes[0])
plot_spatial_feature(
    spots, observed[protein], f"Observed: {protein}", ax=axes[1], vmin=shared_vmin, vmax=shared_vmax
)
plot_spatial_feature(
    spots, predicted[protein], f"Predicted: {protein}", ax=axes[2], vmin=shared_vmin, vmax=shared_vmax
)
residual_scatter = axes[3].scatter(spots["x"], spots["y"], c=residual, cmap="coolwarm", vmin=-limit, vmax=limit, s=24)
axes[3].set(title="Residual (predicted − observed)", xlabel="x", ylabel="y", aspect="equal")
plt.colorbar(residual_scatter, ax=axes[3], fraction=0.046, pad=0.04)
landscape_path = paths.figures / "session03_landscape_comparison.png"
fig.tight_layout(); fig.savefig(landscape_path, dpi=160, bbox_inches="tight"); plt.show()


**How to read it:** spatially localized residuals may reveal tissue boundaries, composition shifts,
antibody effects, or a domain shift. Transcript–protein discordance can be biological because translation,
trafficking, and degradation separate RNA abundance from surface-protein abundance. Observed and predicted
maps share one color scale so intensity differences are visually comparable.


#### Figure 16 — Downstream structure inferred from the full protein panel


In [ ]:
scaled = (predicted - predicted.mean(axis=0)) / (predicted.std(axis=0) + 1e-8)
n_pcs = min(20, scaled.shape[0] - 1, scaled.shape[1])
pcs = PCA(n_components=n_pcs, random_state=7).fit_transform(scaled)
# Arbitrary exploratory k; assess stability before biological claims.
n_clusters = min(6, max(2, len(predicted) // 100))
clusters = KMeans(n_clusters=n_clusters, n_init=20, random_state=7).fit_predict(pcs)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(pcs[:, 0], pcs[:, 1], c=clusters, cmap="tab10", s=14)
axes[0].set(title=f"PCA viz (clustered in {n_pcs} PCs)", xlabel="PC1", ylabel="PC2")
axes[1].scatter(spots["x"], spots["y"], c=clusters, cmap="tab10", s=18)
axes[1].set(title="Inferred-protein clusters in tissue", xlabel="x", ylabel="y", aspect="equal")
embedding_path = paths.figures / "session03_inferred_protein_embedding.png"
fig.tight_layout(); fig.savefig(embedding_path, dpi=160, bbox_inches="tight"); plt.show()


These clusters are hypotheses, not validated cell types. Annotate them only after checking marker panels,
spatial context, robustness to cluster count, and agreement with observed modalities or orthogonal data.


In [ ]:
prompts = (
    "# Session 3 interpretation prompts\n\n"
    "- Which proteins are accurate pointwise but spatially over-smoothed?\n"
    "- Where are residuals spatially localized, and what technical or biological process could explain them?\n"
    "- Which multi-protein clusters are stable and supported by known marker combinations?\n"
    "- What donor, batch, antibody, tissue-boundary, or cell-composition effects could mislead evaluation?\n"
    "- What held-out dataset would best test generalization?\n"
)
prompt_path = paths.results / "session03_interpretation_prompts.md"
prompt_path.write_text(prompts, encoding="utf-8")
manifest = write_checkpoint(
    "3.3", [marker_panel_path, landscape_path, embedding_path, prompt_path],
    summary={"transcript": gene, "protein": protein, "exploratory_clusters": n_clusters, "n_pcs": n_pcs},
    start=paths.root,
)
print(prompts); print(f"Checkpoint written: {manifest}")


### Next steps

A complete analysis now has three evidence layers: per-protein accuracy, spatial fidelity, and biological
interpretation. Before publication, repeat all three on held-out biological samples and include uncertainty
or replicate variability—not only a single fitted map.
